In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 1번

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

%config InlineBackend.figure_format = "retina"

# 시각화 그래프 한글설정
from matplotlib.font_manager import FontProperties, fontManager
font_path = r"C:\Users\Admin\Downloads\JMK체.ttf"
fontManager.addfont(font_path)
font_prop = FontProperties(fname=font_path, size=12)
font_name = font_prop.get_name()

plt.rc('font', family=font_name)
plt.rc('font', size=12)
plt.rc('axes', unicode_minus=False)

In [ ]:
# csv load
df_first = pd.read_csv('c:/ai/downloads/shareData/분양가격/전국 평균 평당 분양가격(2013년 9월부터 2015년 8월까지).csv', encoding='cp949')

In [ ]:
# 데이터 프레임 결측치 확인
df_first.isna().sum()

In [ ]:
# set_index
df_first.set_index('지역', inplace=True)

In [ ]:
# heatmap 시각화
plt.figure(figsize=(20,5))
sns.heatmap(df_first, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.ylabel(None)
plt.show()

# 2번

In [ ]:
# csv load
df_last = pd.read_csv('c:/ai/downloads/shareData/분양가격/주택도시보증공사_전국 신규 민간아파트 분양가격 동향_20260531 (1).csv', encoding='cp949')

In [ ]:
df_last.isna().sum()

In [ ]:
makenan = pd.to_numeric(df_last['분양가격(제곱미터당 천원)'], errors='coerce')
nan_idx = makenan[makenan.isna()].index
df_last.loc[nan_idx, '분양가격(제곱미터당 천원)'].unique()

In [ ]:
df_last[df_last['분양가격(제곱미터당 천원)'].str.strip()=='']

In [ ]:
df_last[df_last['분양가격(제곱미터당 천원)'].str.strip()=='-']

In [ ]:
# 함수 정의
def change_float(row):
    row = row.copy()
    price = row['분양가격(제곱미터당 천원)']
    
    if pd.isna(price):
        row['분양가격(제곱미터당 천원)'] = np.nan
        
    elif price.strip() == '' or price == '-':
        row['분양가격(제곱미터당 천원)'] = np.nan
        
    elif price.find(',') != -1:
        row['분양가격(제곱미터당 천원)'] = float(price.replace(',',''))
        
    else:
        row['분양가격(제곱미터당 천원)'] = float(price)
        
    return row

In [ ]:
# apply 적용
df_last = df_last.apply(change_float, axis=1)

In [ ]:
df_last[df_last['분양가격(제곱미터당 천원)'].isna()].head()

In [ ]:
# 평당분양가격 칼럼 추가
df_last['평당분양가격'] = df_last['분양가격(제곱미터당 천원)'] * 3.3

In [ ]:
# 그 외 이미지 상 변경 점 수정
df_last['규모구분'].unique()

In [ ]:
df_last['전용면적'] = df_last['규모구분'].str.replace('전용면적','').str.replace('이하','').str.replace('제곱미터','m²').str.replace(' ','').str.replace('초과','~')

In [ ]:
df_last = df_last.drop(['규모구분','분양가격(제곱미터당 천원)'], axis=1)

In [ ]:
columns = df_last.pop('전용면적')
df_last.insert(1, '전용면적', columns)

In [ ]:
df_last

# 3번

In [ ]:
# groupby
df_last.groupby(['연도','지역명'])['평당분양가격'].mean().sort_values().unstack().round(1)

# 4번

In [ ]:
# sns 함수 사용
plt.figure(figsize=(20,5))
sns.violinplot(data=df_last, x='연도', y='평당분양가격')
plt.xlabel('연도', fontsize=15)
plt.ylabel('평당분양가격', fontsize=15)

# 5번

In [ ]:
df_first = pd.read_csv('c:/ai/downloads/shareData/분양가격/전국 평균 평당 분양가격(2013년 9월부터 2015년 8월까지).csv', encoding='cp949')

In [ ]:
# 구조 변경
new_first = df_first.melt(id_vars='지역', var_name='연도월', value_name='평당분양가격')
new_first.head()

In [ ]:
# apply를 이용한 데이터 분할
new_first['연도'] = new_first['연도월'].apply(lambda datestr : int(datestr[:4]))
new_first['월'] = new_first['연도월'].apply(lambda datestr : int(datestr[5:-1]))
new_first.head()

In [ ]:
# 필요없는 컬럼 drop
new_first.drop('연도월', axis=1, inplace=True)

In [ ]:
new_first['평당분양가격'] = new_first['평당분양가격'].astype('float64')

In [ ]:
new_first.rename(columns={'지역':'지역명'}, inplace = True)

In [ ]:
new_first.head()

In [ ]:
new_last = df_last[df_last['전용면적']=='모든면적']

In [ ]:
# 필요없는 컬럼 drop
new_last = new_last.drop('전용면적', axis=1)

In [ ]:
new_last.head()

In [ ]:
# concat 사용 및 reset_index, 연도/월 sort
df = pd.concat([new_first, new_last]).sort_values(['연도', '월']).reset_index(drop=True)
df